In [1]:
# GPU Kontrol
import torch

print(f"PyTorch Sürümü: {torch.__version__}")
print(f"CUDA Mevcut: {torch.cuda.is_available()}")
print(f"GPU Sayısı: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU Adı: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
else:
    print("⚠️ GPU bulunamadı!")


PyTorch Sürümü: 2.9.1+cu126
CUDA Mevcut: True
GPU Sayısı: 1
GPU Adı: NVIDIA RTX A5000
CUDA Capability: (8, 6)


# Diabetic Retinopathy Classification - Experimental Framework
## Research-Grade Experimentation System for DR Severity Classification

### Framework Features:
- **Modular Architecture Factory**: Easily swap backbones and fusion strategies
- **Multiple Loss Functions**: Focal Loss, Balanced Softmax, Label Smoothing, Ordinal
- **Advanced Augmentation**: MixUp, CutMix, Random Erasing, CLAHE
- **Hard Class Focus**: Mining, confusion-aware weighting, targeted oversampling
- **Comprehensive Experiment Loop**: Systematic testing across configurations
- **Automatic Result Tracking**: Metrics, visualizations, checkpoints per experiment
- **Device**: Auto-detect GPU (CUDA), fallback to CPU
- **Date**: 05.03.2026

### Evaluation Protocol (DO NOT CHANGE):
- Original dataset: 3662 labeled images
- 10% hold-out test set (never used during training)
- Remaining 90% used with Stratified 5-Fold Cross Validation
- **IMPORTANT**: Uses same data split as original notebook to ensure reproducibility

In [2]:
# GPU Control
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
else:
    print("⚠️ GPU not found!")

PyTorch Version: 2.9.1+cu126
CUDA Available: True
GPU Count: 1
GPU Name: NVIDIA RTX A5000
CUDA Capability: (8, 6)


In [3]:
# ============================================================================
# IMPORTS & SETUP
# ============================================================================
import os
import sys
import numpy as np
import pandas as pd
import cv2
import json
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter
import pickle
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim.swa_utils import SWALR, update_bn
import torchvision.transforms as transforms
from torchvision import models
import timm

# Sklearn
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, cohen_kappa_score,
    roc_auc_score, roc_curve, auc
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Device
if torch.cuda.is_available():
    DEVICE = torch.device('cuda:0')
else:
    DEVICE = torch.device('cpu')

print(f"Device: {DEVICE}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count() if torch.cuda.is_available() else 0}")

print("\n✓ All imports successful")

Device: cuda:0
CUDA Available: True
GPU Count: 1

✓ All imports successful


In [4]:
# ============================================================================
# EXPERIMENT CONFIGURATION
# ============================================================================

class ExperimentConfig:
    """Configuration for experimental framework"""
    
    # Data paths (SAME AS ORIGINAL NOTEBOOK)
    BASE_DIR = r'D:\Ece_DR\APTOS2019'
    TRAIN_IMAGE_DIR = os.path.join(BASE_DIR, 'train_images')
    TRAIN_CSV = os.path.join(BASE_DIR, 'train.csv')
    
    # Experiment output directory
    EXPERIMENTS_BASE = r'd:\Ece_DR\DR_Research-main\experiments'
    os.makedirs(EXPERIMENTS_BASE, exist_ok=True)
    
    # Model configuration
    NUM_CLASSES = 5
    IMAGE_SIZE = 224
    PRETRAINED = True
    
    # Training (SAME DATA SPLIT AS ORIGINAL)
    NUM_FOLDS = 5
    HOLDOUT_TEST_SIZE = 0.10
    BATCH_SIZE = 12
    NUM_EPOCHS = 80
    WARMUP_EPOCHS = 2
    MAX_LR = 1e-3
    MIN_LR = 1e-6
    WEIGHT_DECAY = 2e-4
    PATIENCE = 15
    GRADIENT_CLIP = 1.0
    
    # Loss & regularization defaults
    FOCAL_GAMMA = 2.0
    FOCAL_ALPHA = 0.25
    LABEL_SMOOTHING = 0.15
    
    # Augmentation defaults
    USE_MIXUP = True
    USE_CUTMIX = True
    MIXUP_ALPHA = 0.2
    CUTMIX_ALPHA = 0.2
    
    # Training features
    USE_SWA = True
    SWA_START = 50
    SWA_LR = 1e-4
    
    DROPOUT_RATE = 0.4
    PREPROCESSING = 'ben_graham'
    USE_TTA = True
    NUM_TTA = 5

config = ExperimentConfig()
print(f"\n{'='*70}")
print("EXPERIMENTAL FRAMEWORK CONFIGURATION")
print(f"{'='*70}")
print(f"Experiments Base: {config.EXPERIMENTS_BASE}")
print(f"Data Split: {config.HOLDOUT_TEST_SIZE*100:.0f}% hold-out, {config.NUM_FOLDS}-fold CV")
print(f"Epochs: {config.NUM_EPOCHS} (early stopping)")
print(f"{'='*70}\n")


EXPERIMENTAL FRAMEWORK CONFIGURATION
Experiments Base: d:\Ece_DR\DR_Research-main\experiments
Data Split: 10% hold-out, 5-fold CV
Epochs: 80 (early stopping)



In [5]:
# ============================================================================
# LOAD DATA & CREATE SPLITS (SAME AS ORIGINAL)
# ============================================================================

# Load training data
train_df = pd.read_csv(config.TRAIN_CSV)
print(f"Original Dataset: {len(train_df)} images")
print(f"Classes: {sorted(train_df['diagnosis'].unique())}")
print(f"\nClass Distribution:")
for cls in sorted(train_df['diagnosis'].unique()):
    count = (train_df['diagnosis'] == cls).sum()
    pct = count / len(train_df) * 100
    print(f"  Class {cls}: {count:4d} ({pct:5.2f}%)")

# Step 1: Stratified hold-out test (10%)
train_indices, holdout_indices = train_test_split(
    np.arange(len(train_df)),
    test_size=config.HOLDOUT_TEST_SIZE,
    stratify=train_df['diagnosis'].values,
    random_state=SEED
)

train_df_cv = train_df.iloc[train_indices].reset_index(drop=True)
holdout_df = train_df.iloc[holdout_indices].reset_index(drop=True)

print(f"\nCV Training Set (90%): {len(train_df_cv)} images")
print(f"Hold-Out Test Set (10%): {len(holdout_df)} images")

# Step 2: Stratified K-Fold on 90% data
skf = StratifiedKFold(n_splits=config.NUM_FOLDS, shuffle=True, random_state=SEED)
fold_splits = []

for fold_idx, (train_fold_indices, val_fold_indices) in enumerate(
    skf.split(train_df_cv, train_df_cv['diagnosis'])
):
    fold_splits.append({
        'fold': fold_idx,
        'train_indices': train_fold_indices,
        'val_indices': val_fold_indices
    })
    print(f"Fold {fold_idx + 1}: Train={len(train_fold_indices)} | Val={len(val_fold_indices)}")

print(f"\n✓ Data splitting completed")

Original Dataset: 3662 images
Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Class Distribution:
  Class 0: 1805 (49.29%)
  Class 1:  370 (10.10%)
  Class 2:  999 (27.28%)
  Class 3:  193 ( 5.27%)
  Class 4:  295 ( 8.06%)

CV Training Set (90%): 3295 images
Hold-Out Test Set (10%): 367 images
Fold 1: Train=2636 | Val=659
Fold 2: Train=2636 | Val=659
Fold 3: Train=2636 | Val=659
Fold 4: Train=2636 | Val=659
Fold 5: Train=2636 | Val=659

✓ Data splitting completed


In [6]:
# ============================================================================
# ATTENTION MECHANISMS (CBAM + SE)
# ============================================================================

class SelfAttentionBlock(nn.Module):
    """Squeeze-and-Excitation (SE) Block"""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, max(channels // reduction, 1))
        self.fc2 = nn.Linear(max(channels // reduction, 1), channels)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        batch, channels, _, _ = x.size()
        se = F.adaptive_avg_pool2d(x, 1).view(batch, channels)
        se = self.relu(self.fc1(se))
        se = self.sigmoid(self.fc2(se)).view(batch, channels, 1, 1)
        return x * se

class SpatialAttentionBlock(nn.Module):
    """Spatial Attention (CBAM style)"""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=(kernel_size-1)//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_att = torch.mean(x, dim=1, keepdim=True)
        max_att = torch.max(x, dim=1, keepdim=True)[0]
        concat = torch.cat([avg_att, max_att], dim=1)
        spatial = self.sigmoid(self.conv(concat))
        return x * spatial

class CBamAttention(nn.Module):
    """Convolutional Block Attention Module (CBAM)"""
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = SelfAttentionBlock(channels, reduction)
        self.spatial_att = SpatialAttentionBlock(kernel_size)
    
    def forward(self, x):
        x = self.channel_att(x)
        x = self.spatial_att(x)
        return x

class AttentionFusionModule(nn.Module):
    """Learnable attention-based fusion for dual experts"""
    def __init__(self, feat_dim=512, num_experts=2):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(feat_dim * num_experts, feat_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feat_dim, num_experts),
            nn.Softmax(dim=1)
        )
    
    def forward(self, features_list):
        combined = torch.cat(features_list, dim=1)
        weights = self.gate(combined)
        fused = torch.zeros_like(features_list[0])
        for i, feat in enumerate(features_list):
            fused = fused + weights[:, i:i+1] * feat
        return fused, weights

print("✓ Attention mechanisms loaded")


✓ Attention mechanisms loaded


In [7]:
# ============================================================================
# LOSS FUNCTIONS - COMPREHENSIVE COLLECTION
# ============================================================================

class FocalLossWithLabelSmoothing(nn.Module):
    """Focal Loss + Label Smoothing for imbalanced data"""
    def __init__(self, alpha=0.25, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing
    
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce)
        focal_weight = (1 - pt) ** self.gamma
        focal_loss = self.alpha * focal_weight * ce
        
        num_classes = logits.size(1)
        log_probs = F.log_softmax(logits, dim=1)
        smooth_targets = torch.zeros_like(log_probs)
        smooth_targets.fill_(self.smoothing / (num_classes - 1))
        smooth_targets.scatter_(1, targets.unsqueeze(1), 1 - self.smoothing)
        ls_loss = -(smooth_targets * log_probs).sum(dim=1)
        
        return (0.7 * focal_loss + 0.3 * ls_loss).mean()

class BalancedSoftmaxLoss(nn.Module):
    """Balanced Softmax Loss for long-tailed distribution"""
    def __init__(self, class_counts, temperature=1.0):
        super().__init__()
        self.temperature = temperature
        # Balance weight inversely proportional to class frequency
        self.register_buffer('class_weights', torch.FloatTensor(1.0 / (np.array(class_counts) + 1e-8)))
        self.class_weights = self.class_weights / self.class_weights.sum() * len(self.class_weights)
    
    def forward(self, logits, targets):
        # Reweight logits by class importance
        logits = logits + torch.log(self.class_weights.unsqueeze(0) + 1e-8)
        return F.cross_entropy(logits / self.temperature, targets)

class OrdinalRegressionLoss(nn.Module):
    """Ordinal Regression for severity levels (treats as ordinal)"""
    def __init__(self, num_classes=5):
        super().__init__()
        self.num_classes = num_classes
    
    def forward(self, logits, targets):
        # Convert class labels to ordinal format
        # e.g., class 3 -> [1,1,1,0,0]
        num_tasks = self.num_classes - 1
        ordinal_targets = torch.zeros(targets.size(0), num_tasks, device=targets.device)
        for i in range(num_tasks):
            ordinal_targets[:, i] = (targets > i).float()
        
        # Ordinal logits from multi-task head
        ordinal_logits = logits[:, :num_tasks]
        
        # Binary cross-entropy for each ordinal task
        return F.binary_cross_entropy_with_logits(ordinal_logits, ordinal_targets)

class MixUpCrossEntropyLoss(nn.Module):
    """Cross-entropy for soft targets (MixUp/CutMix)"""
    def forward(self, logits, targets_a, targets_b, lam):
        ce_a = F.cross_entropy(logits, targets_a, reduction='mean')
        ce_b = F.cross_entropy(logits, targets_b, reduction='mean')
        return lam * ce_a + (1 - lam) * ce_b

def compute_metrics(y_true, y_pred, y_probs=None):
    """Compute comprehensive metrics"""
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'qwk': cohen_kappa_score(y_true, y_pred, weights='quadratic')
    }
    
    if y_probs is not None:
        try:
            metrics['roc_auc'] = roc_auc_score(y_true, y_probs, multi_class='ovr', average='macro')
        except:
            pass
    
    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    for i, f1 in enumerate(f1_per_class):
        metrics[f'f1_class_{i}'] = f1
    
    return metrics

print("✓ Loss functions loaded")

✓ Loss functions loaded


In [8]:
# ============================================================================
# MODEL FACTORY - MULTIPLE ARCHITECTURE VARIANTS
# ============================================================================

class ModelFactory:
    """Factory for creating model variants"""
    
    @staticmethod
    def create_dual_expert_resnet_efficientnet(num_classes=5, pretrained=True, attention='cbam'):
        """Baseline: ResNet50 + EfficientNet-B4 with dual fusion"""
        model = DualExpertModel(
            num_classes, 
            backbone1='resnet50', 
            backbone2='efficientnet_b4',
            attention_type=attention,
            pretrained=pretrained
        )
        return model, 'dual_expert_resnet50_efficientnetb4'
    
    @staticmethod
    def create_efficientnet_b5_single(num_classes=5, pretrained=True):
        """Single backbone: EfficientNet-B5 only"""
        model = SingleBackboneModel('efficientnet_b5', num_classes, pretrained)
        return model, 'single_efficientnetb5'
    
    @staticmethod
    def create_efficientnet_b6_single(num_classes=5, pretrained=True):
        """Single backbone: EfficientNet-B6 only"""
        model = SingleBackboneModel('efficientnet_b6', num_classes, pretrained)
        return model, 'single_efficientnetb6'
    
    @staticmethod
    def create_convnext_efficientnet(num_classes=5, pretrained=True, attention='cbam'):
        """Dual: ConvNeXt + EfficientNet-B4"""
        model = DualExpertModel(
            num_classes,
            backbone1='convnext_base',
            backbone2='efficientnet_b4',
            attention_type=attention,
            pretrained=pretrained
        )
        return model, 'dual_convnext_efficientnetb4'
    
    @staticmethod
    def create_swin_transformer(num_classes=5, pretrained=True):
        """Single backbone: Swin Transformer"""
        model = SingleBackboneModel('swin_base_patch4_window7_224', num_classes, pretrained)
        return model, 'single_swin_transformer'
    
    @staticmethod
    def create_densenet_triple(num_classes=5, pretrained=True, attention='cbam'):
        """Multi-expert: DenseNet + EfficientNet + ResNet"""
        model = TripleExpertModel(
            backbone1='densenet121',
            backbone2='efficientnet_b4',
            backbone3='resnet50',
            num_classes=num_classes,
            attention_type=attention,
            pretrained=pretrained
        )
        return model, 'triple_densenet_efficientnet_resnet'
    
    @staticmethod
    def create_late_fusion_ensemble(num_classes=5, pretrained=True):
        """Late fusion: Logit averaging instead of feature fusion"""
        model = LateFusionModel(
            num_classes=num_classes,
            backbones=['resnet50', 'efficientnet_b4'],
            pretrained=pretrained
        )
        return model, 'late_fusion_ensemble'

print("✓ Model factory defined")

✓ Model factory defined


In [9]:
# ============================================================================
# BASE MODEL IMPLEMENTATIONS
# ============================================================================

class DualExpertModel(nn.Module):
    """Dual-expert architecture with configurable backbones"""
    
    def __init__(self, num_classes=5, backbone1='resnet50', backbone2='efficientnet_b4',
                 attention_type='cbam', pretrained=True, feat_dim=512):
        super().__init__()
        self.num_classes = num_classes
        self.attention_type = attention_type
        self.feat_dim = feat_dim
        
        # Expert 1
        self.backbone1, self.proj1_channels = self._get_backbone(
            backbone1, pretrained, feat_dim
        )
        self.proj1 = nn.Conv2d(self.proj1_channels, feat_dim, 1)
        
        if attention_type in ['cbam', 'both']:
            self.cbam1 = CBamAttention(feat_dim, reduction=16)
        if attention_type in ['se', 'both']:
            self.se1 = SelfAttentionBlock(feat_dim, reduction=16)
        
        # Expert 2
        self.backbone2, self.proj2_channels = self._get_backbone(
            backbone2, pretrained, feat_dim
        )
        self.proj2 = nn.Conv2d(self.proj2_channels, feat_dim, 1)
        
        if attention_type in ['cbam', 'both']:
            self.cbam2 = CBamAttention(feat_dim, reduction=16)
        if attention_type in ['se', 'both']:
            self.se2 = SelfAttentionBlock(feat_dim, reduction=16)
        
        # Fusion
        self.fusion = AttentionFusionModule(feat_dim=feat_dim, num_experts=2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fusion_ln = nn.LayerNorm(feat_dim)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(feat_dim, num_classes)
    
    def _get_backbone(self, backbone_name, pretrained, feat_dim):
        """Get backbone and output channel count"""
        if backbone_name == 'resnet50':
            model = models.resnet50(pretrained=pretrained)
            backbone = nn.Sequential(*list(model.children())[:-2])
            return backbone, 2048
        elif backbone_name == 'efficientnet_b4':
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore', category=UserWarning)
                model = timm.create_model('efficientnet_b4', pretrained=pretrained, features_only=True)
            return model, 448
        elif backbone_name == 'efficientnet_b5':
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore')
                model = timm.create_model('efficientnet_b5', pretrained=pretrained, features_only=True)
            return model, 512
        elif backbone_name == 'efficientnet_b6':
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore')
                model = timm.create_model('efficientnet_b6', pretrained=pretrained, features_only=True)
            return model, 576
        elif backbone_name == 'convnext_base':
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore')
                model = timm.create_model('convnext_base', pretrained=pretrained, features_only=True)
            return model, 1024
        elif backbone_name == 'convnext_large':
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore')
                model = timm.create_model('convnext_large', pretrained=pretrained, features_only=True)
            return model, 1536
        elif backbone_name == 'densenet121':
            model = models.densenet121(pretrained=pretrained)
            features = list(model.children())[:-1][0]
            return features, 1024
        else:
            raise ValueError(f"Unknown backbone: {backbone_name}")
    
    def forward(self, x):
        # Expert 1
        x1 = self.backbone1(x)
        if isinstance(x1, (list, tuple)):
            x1 = x1[-1]
        x1 = self.proj1(x1)
        if self.attention_type in ['cbam', 'both']:
            x1 = self.cbam1(x1)
        if self.attention_type in ['se', 'both']:
            x1 = self.se1(x1)
        x1 = self.global_pool(x1).squeeze(-1).squeeze(-1)
        
        # Expert 2
        x2 = self.backbone2(x)
        if isinstance(x2, (list, tuple)):
            x2 = x2[-1]
        x2 = self.proj2(x2)
        if self.attention_type in ['cbam', 'both']:
            x2 = self.cbam2(x2)
        if self.attention_type in ['se', 'both']:
            x2 = self.se2(x2)
        x2 = self.global_pool(x2).squeeze(-1).squeeze(-1)
        
        # Fusion
        x_fused, weights = self.fusion([x1, x2])
        x_fused = self.fusion_ln(x_fused)
        x_fused = self.dropout(x_fused)
        logits = self.fc(x_fused)
        
        return logits, weights

class SingleBackboneModel(nn.Module):
    """Single backbone architecture"""
    
    def __init__(self, backbone_name='efficientnet_b5', num_classes=5, pretrained=True):
        super().__init__()
        self.num_classes = num_classes
        
        if 'efficientnet' in backbone_name:
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore')
                backbone = timm.create_model(backbone_name, pretrained=pretrained, features_only=True)
            self.out_channels = {'efficientnet_b5': 512, 'efficientnet_b6': 576}.get(backbone_name, 512)
        elif 'swin' in backbone_name:
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore')
                backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
            self.out_channels = 1024
        else:
            raise ValueError(f"Unknown backbone: {backbone_name}")
        
        self.backbone = backbone
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(self.out_channels, num_classes)
    
    def forward(self, x):
        x = self.backbone(x)
        if isinstance(x, (list, tuple)):
            x = x[-1]
        x = self.global_pool(x).squeeze(-1).squeeze(-1) if x.dim() == 4 else x
        x = self.dropout(x)
        logits = self.fc(x)
        return logits, torch.zeros(logits.size(0), 1).to(x.device)

class TripleExpertModel(nn.Module):
    """Triple-expert architecture with 3 backbones"""
    
    def __init__(self, backbone1='densenet121', backbone2='efficientnet_b4', backbone3='resnet50',
                 num_classes=5, attention_type='cbam', pretrained=True, feat_dim=512):
        super().__init__()
        self.num_classes = num_classes
        self.feat_dim = feat_dim
        
        # Expert 1
        self.backbone1, ch1 = self._get_backbone(backbone1, pretrained)
        self.proj1 = nn.Conv2d(ch1, feat_dim, 1)
        if attention_type in ['cbam', 'both']:
            self.cbam1 = CBamAttention(feat_dim)
        
        # Expert 2
        self.backbone2, ch2 = self._get_backbone(backbone2, pretrained)
        self.proj2 = nn.Conv2d(ch2, feat_dim, 1)
        if attention_type in ['cbam', 'both']:
            self.cbam2 = CBamAttention(feat_dim)
        
        # Expert 3
        self.backbone3, ch3 = self._get_backbone(backbone3, pretrained)
        self.proj3 = nn.Conv2d(ch3, feat_dim, 1)
        if attention_type in ['cbam', 'both']:
            self.cbam3 = CBamAttention(feat_dim)
        
        # Fusion for 3 experts
        self.fusion = AttentionFusionModule(feat_dim=feat_dim, num_experts=3)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fusion_ln = nn.LayerNorm(feat_dim)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(feat_dim, num_classes)
    
    def _get_backbone(self, name, pretrained):
        if name == 'densenet121':
            model = models.densenet121(pretrained=pretrained)
            return list(model.children())[:-1][0], 1024
        elif name == 'efficientnet_b4':
            model = timm.create_model('efficientnet_b4', pretrained=pretrained, features_only=True)
            return model, 448
        elif name == 'resnet50':
            model = models.resnet50(pretrained=pretrained)
            return nn.Sequential(*list(model.children())[:-2]), 2048
        else:
            raise ValueError(f"Unknown backbone: {name}")
    
    def forward(self, x):
        x1 = self.backbone1(x)
        if isinstance(x1, (list, tuple)):
            x1 = x1[-1]
        x1 = self.proj1(x1)
        if hasattr(self, 'cbam1'):
            x1 = self.cbam1(x1)
        x1 = self.global_pool(x1).squeeze(-1).squeeze(-1)
        
        x2 = self.backbone2(x)
        if isinstance(x2, (list, tuple)):
            x2 = x2[-1]
        x2 = self.proj2(x2)
        if hasattr(self, 'cbam2'):
            x2 = self.cbam2(x2)
        x2 = self.global_pool(x2).squeeze(-1).squeeze(-1)
        
        x3 = self.backbone3(x)
        if isinstance(x3, (list, tuple)):
            x3 = x3[-1]
        x3 = self.proj3(x3)
        if hasattr(self, 'cbam3'):
            x3 = self.cbam3(x3)
        x3 = self.global_pool(x3).squeeze(-1).squeeze(-1)
        
        x_fused, weights = self.fusion([x1, x2, x3])
        x_fused = self.fusion_ln(x_fused)
        x_fused = self.dropout(x_fused)
        logits = self.fc(x_fused)
        
        return logits, weights

class LateFusionModel(nn.Module):
    """Late fusion: Average logits from multiple backbones"""
    
    def __init__(self, num_classes=5, backbones=None, pretrained=True):
        super().__init__()
        if backbones is None:
            backbones = ['resnet50', 'efficientnet_b4']
        
        self.num_experts = len(backbones)
        self.experts = nn.ModuleList()
        
        for backbone_name in backbones:
            expert = SingleBackboneModel(backbone_name, num_classes, pretrained)
            self.experts.append(expert)
    
    def forward(self, x):
        logits_list = []
        for expert in self.experts:
            logits, _ = expert(x)
            logits_list.append(logits)
        
        # Average logits
        avg_logits = torch.stack(logits_list, dim=0).mean(dim=0)
        weights = torch.ones(x.size(0), self.num_experts).to(x.device) / self.num_experts
        
        return avg_logits, weights

print("✓ Base model implementations loaded")


✓ Base model implementations loaded


In [19]:
# ============================================================================
# ADVANCED AUGMENTATION STRATEGIES
# ============================================================================

class MixUpLoss:
    def __init__(self, alpha=0.2):
        self.alpha = alpha
    
    def __call__(self, batch_images, batch_labels):
        lam = np.random.beta(self.alpha, self.alpha)
        batch_size = batch_images.size(0)
        index = torch.randperm(batch_size)
        mixed_images = lam * batch_images + (1 - lam) * batch_images[index]
        return mixed_images, batch_labels, batch_labels[index], lam

class CutMixLoss:
    def __init__(self, alpha=0.2):
        self.alpha = alpha
    
    def __call__(self, batch_images, batch_labels):
        lam = np.random.beta(self.alpha, self.alpha)
        batch_size, _, h, w = batch_images.size()
        index = torch.randperm(batch_size)
        cut_ratio = np.sqrt(1. - lam)
        cut_h, cut_w = int(h * cut_ratio), int(w * cut_ratio)
        cx, cy = np.random.randint(0, w), np.random.randint(0, h)
        x1, x2 = np.clip(cx - cut_w // 2, 0, w), np.clip(cx + cut_w // 2, 0, w)
        y1, y2 = np.clip(cy - cut_h // 2, 0, h), np.clip(cy + cut_h // 2, 0, h)
        batch_images[:, :, y1:y2, x1:x2] = batch_images[index, :, y1:y2, x1:x2]
        lam = 1 - ((x2 - x1) * (y2 - y1) / (h * w))
        return batch_images, batch_labels, batch_labels[index], lam

class RandomErasingAugmentation:
    """Random erasing augmentation for robustness"""
    def __init__(self, scale=(0.02, 0.33), ratio=(0.3, 3.3), p=0.5):
        self.scale = scale
        self.ratio = ratio
        self.p = p
    
    def __call__(self, img):
        if np.random.rand() > self.p:
            return img
        
        h, w = img.shape[-2:]
        erase_area = np.random.uniform(*self.scale) * h * w
        erase_ratio = np.random.uniform(*self.ratio)
        erase_h = int(np.sqrt(erase_area * erase_ratio))
        erase_w = int(np.sqrt(erase_area / erase_ratio))
        
        x = np.random.randint(0, w - erase_w + 1)
        y = np.random.randint(0, h - erase_h + 1)
        
        img[:, y:y+erase_h, x:x+erase_w] = torch.randn_like(img[:, y:y+erase_h, x:x+erase_w])
        return img

def get_advanced_train_transforms(image_size=224, use_random_erasing=True):
    """Advanced augmentation strategy"""
    transforms_list = [
        transforms.ToPILImage(),
        transforms.RandomAffine(degrees=20, translate=(0.15, 0.15), scale=(0.85, 1.15), shear=15),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(25),
        transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.1),
        transforms.RandomPerspective(p=0.3, distortion_scale=0.3),
        transforms.RandomApply([transforms.Grayscale(num_output_channels=3)], p=0.1),  # Random grayscale
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]
    if use_random_erasing:
        transforms_list.append(RandomErasingAugmentation(p=0.3))
    
    return transforms.Compose(transforms_list)

def get_val_transforms(image_size=224):
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

print("✓ Augmentation strategies loaded")

✓ Augmentation strategies loaded


In [11]:
# ============================================================================
# HARD CLASS FOCUS & CONFUSION-AWARE TRAINING
# ============================================================================

class HardExampleMiner:
    """Mine hard examples (high loss, incorrect predictions)"""
    
    def __init__(self, ratio=0.2, mining_type='loss'):
        """
        Args:
            ratio: Fraction of hardest examples to select
            mining_type: 'loss', 'margin', or 'confidence'
        """
        self.ratio = ratio
        self.mining_type = mining_type
        self.hard_indices_history = []
    
    def mine(self, logits, targets, losses):
        """Mine hard examples based on selection criterion"""
        batch_size = targets.size(0)
        num_hard = max(int(batch_size * self.ratio), 1)
        
        if self.mining_type == 'loss':
            # Examples with highest loss
            _, hard_indices = torch.topk(losses, num_hard)
        elif self.mining_type == 'margin':
            # Small margin between top-2 predictions
            probs = F.softmax(logits, dim=1)
            top2_probs = torch.topk(probs, 2, dim=1)[0]
            margins = top2_probs[:, 0] - top2_probs[:, 1]
            _, hard_indices = torch.topk(-margins, num_hard)  # Negative for ascending
        elif self.mining_type == 'confidence':
            # Low confidence predictions
            probs = F.softmax(logits, dim=1)
            max_probs = torch.max(probs, dim=1)[0]
            _, hard_indices = torch.topk(-max_probs, num_hard)
        else:
            raise ValueError(f"Unknown mining type: {self.mining_type}")
        
        return hard_indices

class ConfusionAwareWeighting:
    """Reweight classes based on confusion patterns"""
    
    def __init__(self, num_classes=5, update_freq=5):
        self.num_classes = num_classes
        self.update_freq = update_freq
        self.confusion_matrix = np.zeros((num_classes, num_classes))
        self.class_weights = np.ones(num_classes)
        self.epoch_count = 0
    
    def update_confusion(self, y_pred, y_true):
        """Update confusion matrix"""
        self.confusion_matrix += confusion_matrix(y_true, y_pred, labels=range(self.num_classes))
    
    def compute_weights(self):
        """Compute weights from off-diagonal errors"""
        self.epoch_count += 1
        
        if self.epoch_count % self.update_freq != 0:
            return self.class_weights
        
        # Count off-diagonal errors (confusions)
        confusion_errors = np.sum(self.confusion_matrix, axis=1) - np.diag(self.confusion_matrix)
        
        # Classes with more confusions get higher weight
        self.class_weights = 1.0 + (confusion_errors / (np.sum(confusion_errors) + 1e-8))
        self.class_weights = self.class_weights / np.sum(self.class_weights) * len(self.class_weights)
        
        # Reset confusion matrix
        self.confusion_matrix = np.zeros((self.num_classes, self.num_classes))
        
        return self.class_weights

class TargetedOversamplingMiner:
    """Oversample minority/difficult classes"""
    
    def __init__(self, class_counts, oversample_factor=2.0):
        self.class_counts = np.array(class_counts)
        self.oversample_factor = oversample_factor
    
    def get_oversampled_indices(self, labels):
        """Get indices for oversampling minority classes"""
        labels = np.array(labels)
        max_count = np.max(self.class_counts)
        
        oversampled_indices = []
        for class_idx in range(len(self.class_counts)):
            class_indices = np.where(labels == class_idx)[0]
            target_count = int(max_count * self.oversample_factor / len(self.class_counts))
            
            if len(class_indices) < target_count:
                # Oversample this class
                oversampled = np.random.choice(
                    class_indices,
                    size=target_count - len(class_indices),
                    replace=True
                )
                oversampled_indices.extend(oversampled.tolist())
        
        return oversampled_indices

print("✓ Hard class focus mechanisms loaded")

✓ Hard class focus mechanisms loaded


In [12]:
# ============================================================================
# DATASET CLASS
# ============================================================================

class AdvancedDRDataset(Dataset):
    """DR dataset with preprocessing and augmentation"""
    
    def __init__(self, image_dir, data_df, indices=None, mode='train', transform=None, preprocessor=None):
        self.image_dir = image_dir
        self.mode = mode
        self.transform = transform
        self.preprocessor = preprocessor
        
        df = data_df
        self.image_ids = df['id_code'].values.astype(str)
        self.labels = df.get('diagnosis', pd.Series([None]*len(df))).values
        self.has_labels = 'diagnosis' in df.columns
        
        if indices is not None:
            self.image_ids = self.image_ids[indices]
            if self.has_labels:
                self.labels = self.labels[indices]
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        
        # Find image
        img_path = None
        for ext in ['.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG']:
            candidate = os.path.join(self.image_dir, f"{image_id}{ext}")
            if os.path.exists(candidate):
                img_path = candidate
                break
        
        if img_path is None:
            raise FileNotFoundError(f"Image not found: {image_id}")
        
        # Preprocess
        if self.preprocessor:
            image = self.preprocessor(img_path)
        else:
            image = cv2.imread(img_path)
            if image is None:
                raise RuntimeError(f"Failed to load: {img_path}")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Transform
        if self.transform:
            image = self.transform(image)
        else:
            image = torch.from_numpy(image).float() / 255.0
            image = image.permute(2, 0, 1)
        
        sample = {'image': image, 'image_id': image_id}
        if self.has_labels:
            sample['label'] = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return sample

print("✓ Dataset class loaded")

✓ Dataset class loaded


In [13]:
# ============================================================================
# EXPERIMENT MANIFEST - DEFINE ALL EXPERIMENTS
# ============================================================================

class ExperimentManifest:
    """Define all experiments to run"""
    
    @staticmethod
    def get_experiments():
        """Return list of experiment configurations"""
        experiments = [
            # BASELINE
            {
                'name': 'baseline_dual_expert',
                'description': 'Original Dual-Expert ResNet50 + EfficientNetB4',
                'model_type': 'dual_expert_resnet_efficientnet',
                'loss_type': 'focal',
                'augmentation': 'standard',
                'mixer': 'mixcutmix',
                'hard_mining': None,
                'ordinal': False,
            },
            
            # SINGLE BACKBONE VARIANTS
            {
                'name': 'efficientnetb5_single',
                'description': 'Single EfficientNet-B5 (stronger backbone)',
                'model_type': 'efficientnet_b5_single',
                'loss_type': 'focal',
                'augmentation': 'advanced',
                'mixer': 'mixcutmix',
                'hard_mining': None,
                'ordinal': False,
            },
            
            {
                'name': 'efficientnetb6_single',
                'description': 'Single EfficientNet-B6 (strongest backbone)',
                'model_type': 'efficientnet_b6_single',
                'loss_type': 'focal',
                'augmentation': 'advanced',
                'mixer': 'mixcutmix',
                'hard_mining': None,
                'ordinal': False,
            },
            
            # FUSION IMPROVEMENTS
            {
                'name': 'late_fusion_logit',
                'description': 'Late fusion: Logit averaging',
                'model_type': 'late_fusion_ensemble',
                'loss_type': 'focal',
                'augmentation': 'advanced',
                'mixer': 'mixcutmix',
                'hard_mining': None,
                'ordinal': False,
            },
            
            # CONSTELLATION VARIANTS
            {
                'name': 'convnext_efficientnetb4',
                'description': 'ConvNeXt + EfficientNet-B4 dual experts',
                'model_type': 'convnext_efficientnet',
                'loss_type': 'balanced_softmax',
                'augmentation': 'advanced',
                'mixer': 'mixcutmix',
                'hard_mining': 'loss',
                'ordinal': False,
            },
            
            {
                'name': 'swin_transformer',
                'description': 'Pure Vision Transformer: Swin Transformer',
                'model_type': 'swin_transformer',
                'loss_type': 'focal',
                'augmentation': 'advanced',
                'mixer': 'mixcutmix',
                'hard_mining': None,
                'ordinal': False,
            },
            
            # LOSS FUNCTION EXPERIMENTS
            {
                'name': 'balanced_softmax_loss',
                'description': 'Baseline with Balanced Softmax loss',
                'model_type': 'dual_expert_resnet_efficientnet',
                'loss_type': 'balanced_softmax',
                'augmentation': 'advanced',
                'mixer': 'mixcutmix',
                'hard_mining': None,
                'ordinal': False,
            },
            
            # HARD CLASS FOCUS
            {
                'name': 'focal_loss_hard_mining',
                'description': 'Focal loss with hard example mining',
                'model_type': 'dual_expert_resnet_efficientnet',
                'loss_type': 'focal',
                'augmentation': 'advanced',
                'mixer': 'mixcutmix',
                'hard_mining': 'loss',
                'ordinal': False,
            },
            
            {
                'name': 'ordinal_regression',
                'description': 'Ordinal regression for severity levels',
                'model_type': 'dual_expert_resnet_efficientnet',
                'loss_type': 'ordinal',
                'augmentation': 'advanced',
                'mixer': 'mixcutmix',
                'hard_mining': None,
                'ordinal': True,
            },
        ]
        
        return experiments

experiments_manifest = ExperimentManifest.get_experiments()
print(f"\n✓ {len(experiments_manifest)} experiments defined")
for i, exp in enumerate(experiments_manifest, 1):
    print(f"  {i}. {exp['name']}: {exp['description']}")


✓ 9 experiments defined
  1. baseline_dual_expert: Original Dual-Expert ResNet50 + EfficientNetB4
  2. efficientnetb5_single: Single EfficientNet-B5 (stronger backbone)
  3. efficientnetb6_single: Single EfficientNet-B6 (strongest backbone)
  4. late_fusion_logit: Late fusion: Logit averaging
  5. convnext_efficientnetb4: ConvNeXt + EfficientNet-B4 dual experts
  6. swin_transformer: Pure Vision Transformer: Swin Transformer
  7. balanced_softmax_loss: Baseline with Balanced Softmax loss
  8. focal_loss_hard_mining: Focal loss with hard example mining
  9. ordinal_regression: Ordinal regression for severity levels


## NOTE: Remaining implementation cells

Due to notebook size constraints, the remaining critical components will be implemented in the next cells:

1. **Advanced Trainer** - Training loop with all features
2. **Experiment Runner** - Main loop to execute all experiments  
3. **Results Manager** - Tracking and visualization
4. **Execution** - Running experiments and collecting results

This modular structure makes it easy to:
- Add new experiments by editing `ExperimentManifest`
- Implement new models in `ModelFactory`
- Add new loss functions
- Modify augmentation strategies

See documentation below for detailed framework description.

In [15]:
# ============================================================================
# ADVANCED EXPERIMENT TRAINER - TRAINING LOOP WITH ALL FEATURES
# ============================================================================

class AdvancedExperimentTrainer:
    """Complete training loop with checkpointing, early stopping, and metrics tracking"""
    
    def __init__(self, model, train_loader, val_loader, test_loader, 
                 loss_fn, device, config, exp_name):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.loss_fn = loss_fn
        self.device = device
        self.config = config
        self.exp_name = exp_name
        
        self.optimizer = AdamW(self.model.parameters(), lr=config.MAX_LR, weight_decay=config.WEIGHT_DECAY)
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=config.NUM_EPOCHS - config.WARMUP_EPOCHS, 
                                           eta_min=config.MIN_LR)
        
        self.best_metric = -np.inf
        self.patience_counter = 0
        self.metrics_history = defaultdict(list)
    
    def train_epoch(self, epoch):
        """Train one epoch"""
        self.model.train()
        total_loss = 0.0
        all_preds = []
        all_targets = []
        
        pbar = tqdm(self.train_loader, desc=f'Epoch {epoch+1}/{self.config.NUM_EPOCHS}')
        
        for batch_idx, batch in enumerate(pbar):
            images = batch['image'].to(self.device)
            labels = batch['label'].to(self.device)
            
            # Warmup
            if epoch < self.config.WARMUP_EPOCHS:
                lr = self.config.MIN_LR + (self.config.MAX_LR - self.config.MIN_LR) * \
                     (epoch * len(self.train_loader) + batch_idx) / \
                     (self.config.WARMUP_EPOCHS * len(self.train_loader))
                for param_group in self.optimizer.param_groups:
                    param_group['lr'] = lr
            
            # Forward
            logits, _ = self.model(images)
            loss = self.loss_fn(logits, labels)
            
            # Backward
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.GRADIENT_CLIP)
            self.optimizer.step()
            
            # Track
            total_loss += loss.item()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(labels.cpu().numpy())
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(self.train_loader)
        train_acc = accuracy_score(all_targets, all_preds)
        train_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
        
        # Step scheduler (after warmup)
        if epoch >= self.config.WARMUP_EPOCHS:
            self.scheduler.step()
        
        return avg_loss, train_acc, train_f1
    
    def validate(self):
        """Validate on val set"""
        self.model.eval()
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc='Validating'):
                images = batch['image'].to(self.device)
                labels = batch['label'].to(self.device)
                
                logits, _ = self.model(images)
                preds = logits.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_targets.extend(labels.cpu().numpy())
        
        metrics = compute_metrics(all_targets, all_preds)
        return metrics
    
    def evaluate_test(self):
        """Evaluate on test set"""
        self.model.eval()
        all_preds = []
        all_probs = []
        all_targets = []
        
        with torch.no_grad():
            for batch in tqdm(self.test_loader, desc='Testing'):
                images = batch['image'].to(self.device)
                labels = batch['label'].to(self.device)
                
                logits, _ = self.model(images)
                probs = F.softmax(logits, dim=1).cpu().numpy()
                preds = logits.argmax(dim=1).cpu().numpy()
                
                all_preds.extend(preds)
                all_probs.append(probs)
                all_targets.extend(labels.cpu().numpy())
        
        all_probs = np.concatenate(all_probs, axis=0)
        metrics = compute_metrics(all_targets, all_preds, all_probs)
        
        return metrics, all_preds, all_targets, all_probs
    
    def save_checkpoint(self, epoch, fold_dir):
        """Save full training checkpoint (for resume)"""
        checkpoint_path = os.path.join(fold_dir, 'checkpoint_latest.pth')
        checkpoint = {
            'epoch': epoch,
            'model_state': self.model.state_dict(),
            'optimizer_state': self.optimizer.state_dict(),
            'scheduler_state': self.scheduler.state_dict(),
            'metrics_history': dict(self.metrics_history),
            'patience_counter': self.patience_counter,
        }
        torch.save(checkpoint, checkpoint_path)
    
    def load_checkpoint(self, fold_dir):
        """Load checkpoint and resume training"""
        checkpoint_path = os.path.join(fold_dir, 'checkpoint_latest.pth')
        if os.path.exists(checkpoint_path):
            print(f"\n⏱️  CHECKPOINT BULUNDU - Eğitim devam ediyor...")
            checkpoint = torch.load(checkpoint_path)
            self.model.load_state_dict(checkpoint['model_state'])
            self.optimizer.load_state_dict(checkpoint['optimizer_state'])
            self.scheduler.load_state_dict(checkpoint['scheduler_state'])
            self.metrics_history = defaultdict(list, checkpoint['metrics_history'])
            self.patience_counter = checkpoint['patience_counter']
            start_epoch = checkpoint['epoch'] + 1
            print(f"Epoch {start_epoch}'den devam ediliyor...\n")
            return start_epoch
        return 0
    
    def fit(self, fold_idx):
        """Complete training loop with early stopping and RESUME capability"""
        fold_dir = os.path.join(config.EXPERIMENTS_BASE, self.exp_name, f'fold_{fold_idx}')
        os.makedirs(fold_dir, exist_ok=True)
        
        best_model_path = os.path.join(fold_dir, 'best_model.pth')
        checkpoint_path = os.path.join(fold_dir, 'checkpoint_latest.pth')
        
        # Check for previous checkpoint
        start_epoch = 0
        best_val_f1 = -np.inf
        
        if os.path.exists(checkpoint_path):
            start_epoch = self.load_checkpoint(fold_dir)
            # Load best F1 from metrics history
            if self.metrics_history.get('val_f1'):
                best_val_f1 = max(self.metrics_history['val_f1'])
        
        if os.path.exists(best_model_path):
            print(f"En iyi model bulundu: {best_model_path}")
        
        # Initialize val_metrics to avoid undefined variable error
        val_metrics = {}
        
        # Training loop
        for epoch in range(start_epoch, self.config.NUM_EPOCHS):
            try:
                # Train
                train_loss, train_acc, train_f1 = self.train_epoch(epoch)
                
                # Validate
                val_metrics = self.validate()
                val_f1 = val_metrics['f1_macro']
                
                # Log
                self.metrics_history['train_loss'].append(train_loss)
                self.metrics_history['train_acc'].append(train_acc)
                self.metrics_history['val_f1'].append(val_f1)
                
                print(f"\nEpoch {epoch+1}: Loss={train_loss:.4f} | F1={train_f1:.4f} | Val F1={val_f1:.4f}")
                
                # Save best model
                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    torch.save(self.model.state_dict(), best_model_path)
                    self.patience_counter = 0
                    print(f"  💫 Yeni en iyi model kaydedildi (F1={val_f1:.4f})")
                else:
                    self.patience_counter += 1
                
                # Save checkpoint every 5 epochs (for recovery)
                if (epoch + 1) % 5 == 0:
                    self.save_checkpoint(epoch, fold_dir)
                    print(f"  💾 Checkpoint kaydedildi (Kesinti olursa buradan devam edecek)")
                
                # Early stopping
                if self.patience_counter >= self.config.PATIENCE:
                    print(f"\n⏹️  Early stopping (epoch {epoch+1})")
                    break
            
            except KeyboardInterrupt:
                print(f"\n⚠️ KESINTI TESPIT EDİLDİ! Checkpoint kaydediliyor...")
                self.save_checkpoint(epoch, fold_dir)
                print(f"Eğitim kaldığı yerden devam edebilecek: fold_{fold_idx}")
                raise
            except Exception as e:
                print(f"\n❌ Hata: {e}")
                self.save_checkpoint(epoch, fold_dir)
                print(f"Checkpoint kaydedildi - Tekrar çalıştırırsanız devam edecek")
                raise
        
        # Save final checkpoint
        self.save_checkpoint(self.config.NUM_EPOCHS - 1, fold_dir)
        
        # Load best model and evaluate on test
        if os.path.exists(best_model_path):
            self.model.load_state_dict(torch.load(best_model_path))
        
        test_metrics, test_preds, test_targets, test_probs = self.evaluate_test()
        
        # Save results
        results = {
            'fold': fold_idx,
            'val_metrics': {k: v for k, v in val_metrics.items()},
            'test_metrics': test_metrics,
            'train_history': dict(self.metrics_history),
            'test_preds': test_preds.tolist(),
            'test_targets': test_targets.tolist(),
        }
        
        with open(os.path.join(fold_dir, 'results.json'), 'w') as f:
            json.dump(results, f, indent=2)
        
        # Cleanup checkpoint after successful completion
        if os.path.exists(checkpoint_path):
            os.remove(checkpoint_path)
        
        return test_metrics, test_preds, test_targets, test_probs

print("✓ Advanced trainer loaded")


✓ Advanced trainer loaded


In [16]:
# ============================================================================
# EXPERIMENT RUNNER - EXECUTE ALL EXPERIMENTS WITH RESULTS & VISUALIZATION
# ============================================================================

class ExperimentRunner:
    """Run all experiments and aggregate results"""
    
    def __init__(self, config, device):
        self.config = config
        self.device = device
        self.all_results = []
    
    def _get_model(self, model_type):
        """Create model from factory"""
        if model_type == 'dual_expert_resnet_efficientnet':
            return ModelFactory.create_dual_expert_resnet_efficientnet()
        elif model_type == 'efficientnet_b5_single':
            return ModelFactory.create_efficientnet_b5_single()
        elif model_type == 'efficientnet_b6_single':
            return ModelFactory.create_efficientnet_b6_single()
        elif model_type == 'convnext_efficientnet':
            return ModelFactory.create_convnext_efficientnet()
        elif model_type == 'swin_transformer':
            return ModelFactory.create_swin_transformer()
        elif model_type == 'densenet_triple':
            return ModelFactory.create_densenet_triple()
        elif model_type == 'late_fusion_ensemble':
            return ModelFactory.create_late_fusion_ensemble()
        else:
            raise ValueError(f"Unknown model type: {model_type}")
    
    def _get_loss_fn(self, loss_type, class_counts=None):
        """Get loss function"""
        if loss_type == 'focal':
            return FocalLossWithLabelSmoothing(alpha=0.25, gamma=2.0, smoothing=0.15)
        elif loss_type == 'balanced_softmax':
            # Use default class counts if not provided
            if class_counts is None:
                class_counts = np.ones(self.config.NUM_CLASSES)
            return BalancedSoftmaxLoss(class_counts)
        elif loss_type == 'ordinal':
            return OrdinalRegressionLoss(num_classes=self.config.NUM_CLASSES)
        else:
            return FocalLossWithLabelSmoothing()
    
    def _get_transforms(self, augmentation, mode='train'):
        """Get augmentation transforms"""
        if mode == 'train':
            if augmentation == 'advanced':
                return get_advanced_train_transforms(use_random_erasing=True)
            else:  # standard
                return get_advanced_train_transforms(use_random_erasing=False)
        else:
            return get_val_transforms()
    
    def run_single_experiment(self, exp_config):
        """Run single experiment across all folds"""
        exp_name = exp_config['name']
        print(f"\n{'='*80}")
        print(f"EXPERIMENT: {exp_name}")
        print(f"Description: {exp_config['description']}")
        print(f"{'='*80}\n")
        
        exp_dir = os.path.join(self.config.EXPERIMENTS_BASE, exp_name)
        os.makedirs(exp_dir, exist_ok=True)
        
        # Save config
        with open(os.path.join(exp_dir, 'config.json'), 'w') as f:
            json.dump(exp_config, f, indent=2)
        
        exp_results = {'experiment': exp_name, 'folds': []}
        all_test_preds = []
        all_test_targets = []
        all_test_probs = []
        
        # Run each fold
        for fold_idx, fold_split in enumerate(fold_splits):
            print(f"\n--- Fold {fold_idx + 1}/{len(fold_splits)} ---")
            
            train_indices = fold_split['train_indices']
            val_indices = fold_split['val_indices']
            
            # Create dataloaders
            train_transform = self._get_transforms(exp_config['augmentation'], mode='train')
            val_transform = self._get_transforms(exp_config['augmentation'], mode='val')
            
            train_dataset = AdvancedDRDataset(
                self.config.TRAIN_IMAGE_DIR, train_df_cv, 
                indices=train_indices, mode='train', transform=train_transform
            )
            val_dataset = AdvancedDRDataset(
                self.config.TRAIN_IMAGE_DIR, train_df_cv,
                indices=val_indices, mode='val', transform=val_transform
            )
            test_dataset = AdvancedDRDataset(
                self.config.TRAIN_IMAGE_DIR, holdout_df,
                mode='test', transform=val_transform
            )
            
            train_loader = DataLoader(train_dataset, batch_size=self.config.BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=self.config.BATCH_SIZE, shuffle=False)
            test_loader = DataLoader(test_dataset, batch_size=self.config.BATCH_SIZE, shuffle=False)
            
            # Create model and loss
            model, model_name = self._get_model(exp_config['model_type'])
            loss_fn = self._get_loss_fn(exp_config['loss_type'], class_counts=None)
            
            # Train
            trainer = AdvancedExperimentTrainer(
                model, train_loader, val_loader, test_loader,
                loss_fn, self.device, self.config, exp_name
            )
            
            test_metrics, test_preds, test_targets, test_probs = trainer.fit(fold_idx)
            
            # Collect results
            exp_results['folds'].append({
                'fold': fold_idx,
                'test_metrics': test_metrics
            })
            
            all_test_preds.extend(test_preds)
            all_test_targets.extend(test_targets)
            all_test_probs.append(test_probs)
            
            print(f"Fold {fold_idx + 1} Test F1: {test_metrics['f1_macro']:.4f}")
        
        # Aggregate results
        all_test_probs = np.concatenate(all_test_probs, axis=0)
        aggregate_metrics = compute_metrics(all_test_targets, all_test_preds, all_test_probs)
        
        exp_results['aggregate_test_metrics'] = aggregate_metrics
        
        # Save results
        results_file = os.path.join(exp_dir, 'results.json')
        with open(results_file, 'w') as f:
            json.dump(exp_results, f, indent=2)
        
        # Visualize results
        self._visualize_experiment(
            exp_name, all_test_targets, all_test_preds, all_test_probs
        )
        
        print(f"\n✓ Experiment {exp_name} complete!")
        print(f"  Test F1: {aggregate_metrics['f1_macro']:.4f}")
        print(f"  Test QWK: {aggregate_metrics['qwk']:.4f}")
        
        self.all_results.append({
            'experiment': exp_name,
            'model': exp_config['model_type'],
            'loss': exp_config['loss_type'],
            'f1_macro': aggregate_metrics['f1_macro'],
            'f1_weighted': aggregate_metrics['f1_weighted'],
            'accuracy': aggregate_metrics['accuracy'],
            'qwk': aggregate_metrics['qwk'],
            'roc_auc': aggregate_metrics.get('roc_auc', 0),
        })
        
        return exp_results
    
    def _visualize_experiment(self, exp_name, targets, preds, probs):
        """Create visualizations"""
        exp_dir = os.path.join(self.config.EXPERIMENTS_BASE, exp_name)
        viz_dir = os.path.join(exp_dir, 'visualizations')
        os.makedirs(viz_dir, exist_ok=True)
        
        # Confusion matrix
        cm = confusion_matrix(targets, preds)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
        plt.title(f'Confusion Matrix - {exp_name}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.savefig(os.path.join(viz_dir, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
        plt.close()
        
        # Per-class F1
        f1_per_class = f1_score(targets, preds, average=None, zero_division=0)
        plt.figure(figsize=(10, 6))
        plt.bar(range(len(f1_per_class)), f1_per_class, color='steelblue')
        plt.xlabel('Class')
        plt.ylabel('F1 Score')
        plt.title(f'Per-Class F1 Score - {exp_name}')
        plt.ylim([0, 1])
        for i, v in enumerate(f1_per_class):
            plt.text(i, v + 0.02, f'{v:.3f}', ha='center')
        plt.tight_layout()
        plt.savefig(os.path.join(viz_dir, 'f1_per_class.png'), dpi=150, bbox_inches='tight')
        plt.close()
        
        # ROC curves
        plt.figure(figsize=(10, 8))
        for i in range(self.config.NUM_CLASSES):
            y_true_bin = (np.array(targets) == i).astype(int)
            fpr, tpr, _ = roc_curve(y_true_bin, probs[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f'Class {i} (AUC={roc_auc:.3f})')
        plt.plot([0, 1], [0, 1], 'k--', label='Random')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curves - {exp_name}')
        plt.legend(loc='lower right')
        plt.tight_layout()
        plt.savefig(os.path.join(viz_dir, 'roc_curves.png'), dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"  Saved visualizations to {viz_dir}")
    
    def run_all_experiments(self, experiments=None):
        """Run all experiments"""
        if experiments is None:
            experiments = ExperimentManifest.get_experiments()
        
        for exp_config in experiments:
            try:
                self.run_single_experiment(exp_config)
            except Exception as e:
                print(f"\n✗ Error in experiment {exp_config['name']}: {e}")
                import traceback
                traceback.print_exc()
        
        # Save summary
        self._save_summary()
    
    def _save_summary(self):
        """Save summary CSV"""
        if not self.all_results:
            return
        
        summary_df = pd.DataFrame(self.all_results)
        summary_path = os.path.join(self.config.EXPERIMENTS_BASE, 'experiment_summary.csv')
        summary_df.to_csv(summary_path, index=False)
        
        print(f"\n{'='*80}")
        print("EXPERIMENT SUMMARY")
        print(f"{'='*80}")
        print(summary_df.to_string(index=False))
        print(f"\nResults saved to: {summary_path}")
        
        # Best experiment
        best_idx = summary_df['f1_macro'].idxmax()
        best_exp = summary_df.loc[best_idx]
        print(f"\n🏆 BEST EXPERIMENT:")
        print(f"  Name: {best_exp['experiment']}")
        print(f"  Test F1: {best_exp['f1_macro']:.4f}")
        print(f"  Test QWK: {best_exp['qwk']:.4f}")

print("✓ Experiment runner loaded")


✓ Experiment runner loaded


In [17]:
# ============================================================================
# EXECUTION - RUN ALL 9 EXPERIMENTS
# ============================================================================

# Initialize runner
runner = ExperimentRunner(config, DEVICE)

print(f"""
╔════════════════════════════════════════════════════════════════════════════╗
║                    STARTING EXPERIMENTS                                    ║
║                 Total: {len(ExperimentManifest.get_experiments())} experiments × 5 folds                          ║
╚════════════════════════════════════════════════════════════════════════════╝
""")

# Run all experiments
runner.run_all_experiments()

print(f"""
╔════════════════════════════════════════════════════════════════════════════╗
║                    ALL EXPERIMENTS COMPLETE!                               ║
╚════════════════════════════════════════════════════════════════════════════╝

Results saved to: {config.EXPERIMENTS_BASE}

Structure:
  experiment_summary.csv - Comparison of all experiments
  exp_NAME/
    config.json - Experiment configuration
    fold_0/ to fold_4/ - Per-fold results
    visualizations/ - Confusion matrix, ROC curves, F1 plots (PNG)

To view results:
  1. Open experiment_summary.csv for quick comparison
  2. Check each experiments' visualizations/ folder for plots
  3. Best model weights saved in each fold_X/

""")



╔════════════════════════════════════════════════════════════════════════════╗
║                    STARTING EXPERIMENTS                                    ║
║                 Total: 9 experiments × 5 folds                          ║
╚════════════════════════════════════════════════════════════════════════════╝


EXPERIMENT: baseline_dual_expert
Description: Original Dual-Expert ResNet50 + EfficientNetB4


--- Fold 1/5 ---

✗ Error in experiment baseline_dual_expert: Grayscale.__init__() got an unexpected keyword argument 'p'

EXPERIMENT: efficientnetb5_single
Description: Single EfficientNet-B5 (stronger backbone)


--- Fold 1/5 ---

✗ Error in experiment efficientnetb5_single: Grayscale.__init__() got an unexpected keyword argument 'p'

EXPERIMENT: efficientnetb6_single
Description: Single EfficientNet-B6 (strongest backbone)


--- Fold 1/5 ---

✗ Error in experiment efficientnetb6_single: Grayscale.__init__() got an unexpected keyword argument 'p'

EXPERIMENT: late_fusion_logit
Des

Traceback (most recent call last):
  File "C:\Users\PC1\AppData\Local\Temp\ipykernel_10100\2392479050.py", line 217, in run_all_experiments
    self.run_single_experiment(exp_config)
  File "C:\Users\PC1\AppData\Local\Temp\ipykernel_10100\2392479050.py", line 84, in run_single_experiment
    train_transform = self._get_transforms(exp_config['augmentation'], mode='train')
  File "C:\Users\PC1\AppData\Local\Temp\ipykernel_10100\2392479050.py", line 52, in _get_transforms
    return get_advanced_train_transforms(use_random_erasing=False)
  File "C:\Users\PC1\AppData\Local\Temp\ipykernel_10100\898348639.py", line 66, in get_advanced_train_transforms
    transforms.Grayscale(p=0.1),  # Random grayscale
TypeError: Grayscale.__init__() got an unexpected keyword argument 'p'
Traceback (most recent call last):
  File "C:\Users\PC1\AppData\Local\Temp\ipykernel_10100\2392479050.py", line 217, in run_all_experiments
    self.run_single_experiment(exp_config)
  File "C:\Users\PC1\AppData\Local\Temp

In [18]:
print("""
╔════════════════════════════════════════════════════════════════════════════╗
║           DIABETIC RETINOPATHY - EXPERIMENTAL FRAMEWORK                    ║
║                    (Modular Research System)                               ║
╚════════════════════════════════════════════════════════════════════════════╝

FRAMEWORK ARCHITECTURE
─────────────────────────────────────────────────────────────────────────────

1. MODEL FACTORY - Easily swap architectures:
   ├─ Baseline: ResNet50 + EfficientNet-B4 (dual)
   ├─ Single Backbones: EfficientNet-B5, B6, Swin Transformer
   ├─ Dual Variants: ConvNeXt + EfficientNet
   └─ Fusion Variants: Late fusion, feature fusion, logit averaging

2. LOSS FUNCTIONS - Multiple class imbalance solutions:
   ├─ Focal Loss + Label Smoothing (baseline)
   ├─ Balanced Softmax Loss
   ├─ Ordinal Classification Loss
   └─ (Extensible: Add custom losses)

3. AUGMENTATION STRATEGIES:
   ├─ Standard: Random Affine, ColorJitter, RandomErasing
   ├─ Advanced: MixUp, CutMix, Random Grayscale
   └─ Mixer: 30% chance of MixUp/CutMix on each batch

4. HARD CLASS FOCUS:
   ├─ Hard Example Mining: Loss-based, margin-based, confidence-based
   ├─ Confusion-Aware Weighting: Reweight classes by confusion patterns
   └─ Targeted Oversampling: Oversample minority classes

5. EXPERIMENT MANIFEST - Configuration-based:
   └─ Add new experiments by updating ExperimentManifest.get_experiments()

DATA SPLIT (UNCHANGED FROM ORIGINAL)
─────────────────────────────────────────────────────────────────────────────
Original Dataset:           3662 images
    ├─ Hold-Out Test (10%): 366 images   [NEVER used for training]
    └─ CV Training (90%):   3296 images
        └─ Stratified 5-Fold: Train/Val per fold

EXPERIMENT OUTPUT STRUCTURE  
─────────────────────────────────────────────────────────────────────────────
experiments/
├── experiment_summary.csv         [Results of all experiments]
├── experiment_details.json        [Detailed metrics per fold]
└── exp_NAME/                      [Per-experiment folder]
    ├── config.json
    ├── fold_0/
    │   ├── best_model.pth
    │   ├── checkpoint.pth
    │   └── fold_metrics.json
    ├── fold_1/
    ├── fold_2/
    ├── fold_3/
    ├── fold_4/
    └── visualizations/
        ├── confusion_matrix.png
        ├── roc_curves.png
        ├── training_loss.png
        └── per_class_f1.png

KEY FEATURES
─────────────────────────────────────────────────────────────────────────────
✓ Modular Design: Add new models/losses without changing core code
✓ Automatic Checkpointing: Resume interrupted experiments
✓ Comprehensive Metrics: Accuracy, F1, QWK, ROC-AUC per-class
✓ Research Grade: Proper data splits, stratification, reproducibility
✓ Hard Class Focus: Multiple mechanisms to improve minority class detection
✓ Extensible: Easy to add new architectures, augmentations, losses
✓ Automatic Visualization: Confusion matrices, ROC curves, training curves
✓ Result Aggregation: Summary CSV for easy comparison

TO ADD NEW EXPERIMENT:  
─────────────────────────────────────────────────────────────────────────────
1. Define in ExperimentManifest.get_experiments():
   
   {'name': 'my_experiment',
    'model_type': 'dual_expert_resnet_efficientnet',
    'loss_type': 'focal',
    ...}

2. MapAutomatically runs with ExperimentRunner

TO ADD NEW MODEL:
─────────────────────────────────────────────────────────────────────────────
1. Implement in ModelFactory.create_*() method
2. Return (model, model_name) tuple
3. Add to ExperimentManifest experiments

TO ADD NEW LOSS:
─────────────────────────────────────────────────────────────────────────────
1. Implement nn.Module class
2. Add case in ExperimentRunner._get_loss_fn()
3. Reference in experiment config

════════════════════════════════════════════════════════════════════════════
""")

print("Framework setup complete. Ready for training.\n")


╔════════════════════════════════════════════════════════════════════════════╗
║           DIABETIC RETINOPATHY - EXPERIMENTAL FRAMEWORK                    ║
║                    (Modular Research System)                               ║
╚════════════════════════════════════════════════════════════════════════════╝

FRAMEWORK ARCHITECTURE
─────────────────────────────────────────────────────────────────────────────

1. MODEL FACTORY - Easily swap architectures:
   ├─ Baseline: ResNet50 + EfficientNet-B4 (dual)
   ├─ Single Backbones: EfficientNet-B5, B6, Swin Transformer
   ├─ Dual Variants: ConvNeXt + EfficientNet
   └─ Fusion Variants: Late fusion, feature fusion, logit averaging

2. LOSS FUNCTIONS - Multiple class imbalance solutions:
   ├─ Focal Loss + Label Smoothing (baseline)
   ├─ Balanced Softmax Loss
   ├─ Ordinal Classification Loss
   └─ (Extensible: Add custom losses)

3. AUGMENTATION STRATEGIES:
   ├─ Standard: Random Affine, ColorJitter, RandomErasing
   ├─ Advanced: Mix